In [1]:
from evaluation_utils import evaluate_iplg_convergence
import GridMLM_tokenizers
from GridMLM_tokenizers import CSGridMLMTokenizer
from data_utils import CSGridMLMDataset
from torch.utils.data import DataLoader
from models import SEFiLMModel
import torch
from torch.nn import CrossEntropyLoss
import os
import pickle
from train_utils import train_IPLG
from data_utils import latent_MH_collate_fn
from pprint import pprint
import pandas as pd

device_name = 'cuda:0'
batch_size = 4

/home/maximos/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
home_path = f'data/latent_datasets/SE/nottingham_test.pickle'
foreign_path = f'data/latent_datasets/SE/gjt_CA_test.pickle'

In [3]:
with open(home_path, 'rb') as f:
    home_dataset = pickle.load(f)
home_loader = DataLoader(
    home_dataset,
    batch_size=batch_size,
    shuffle=False,
    drop_last=True,
    collate_fn=latent_MH_collate_fn
)
with open(foreign_path, 'rb') as f:
    foreign_dataset = pickle.load(f)
foreign_loader = DataLoader(
    foreign_dataset,
    batch_size=batch_size,
    shuffle=False,
    drop_last=True,
    collate_fn=latent_MH_collate_fn
)

print(len(home_loader), len(foreign_loader))

if device_name == 'cpu':
    device = torch.device('cpu')
else:
    if torch.cuda.is_available():
        device = torch.device(device_name)
    else:
        print('Selected device not available: ' + device_name)
# end device selection

latent_loss_fn = torch.nn.MSELoss()

tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)
mask_token_id = tokenizer.mask_token_id
bar_token_id = tokenizer.bar_token_id

logits_loss_fn =CrossEntropyLoss(ignore_index=-100)


11 7


In [4]:
loss_scheme = 'l'

d_model = 512
transformer_model = SEFiLMModel(
    chord_vocab_size=len(tokenizer.vocab),
    d_model=d_model,
    nhead=8,
    num_layers=8,
    grid_length=80,
    pianoroll_dim=tokenizer.pianoroll_dim,
    guidance_dim=d_model,
    device=device,
)
checkpoint = torch.load(f'saved_models/iplg/SE/iplg_{loss_scheme}_loss.pt', map_location=device_name)
transformer_model.load_state_dict(checkpoint)
transformer_model.to(device)
transformer_model.eval()

SEFiLMModel(
  (melody_proj): Linear(in_features=13, out_features=512, bias=True)
  (harmony_embedding): Embedding(355, 512)
  (dropout): Dropout(p=0.3, inplace=False)
  (encoder_layers): ModuleList(
    (0-7): 8 x TransformerEncoderLayerWithFiLM(
      (layer): TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (linear1): Linear(in_features=512, out_features=2048, bias=True)
        (dropout): Dropout(p=0.3, inplace=False)
        (linear2): Linear(in_features=2048, out_features=512, bias=True)
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.3, inplace=False)
        (dropout2): Dropout(p=0.3, inplace=False)
      )
      (film): FiLMAdapter(
        (gamma): Linear(in_features=512, out_features=512, bias=True)
        (beta): Lin

In [5]:
eval_latent, eval_logits, eval_unique, eval_conf = evaluate_iplg_convergence(
    transformer_model,
    home_loader,
    foreign_loader,
    logits_loss_fn,
    latent_loss_fn,
    mask_token_id,
    bar_token_id,
    device,
    interpolate=False,
    extrapolate=False
)

Epoch 0@0| val: 100%|██████████| 11/11 [00:00<00:00, 66.10batch/s, acc=0.773, floss=0.324, hloss=0.224, loss=0.33]
Epoch 0@0| val: 100%|██████████| 11/11 [00:00<00:00, 138.26batch/s, acc=0.773, floss=0.33, hloss=0.224, loss=0.33]
Epoch 0@0| val: 100%|██████████| 11/11 [00:00<00:00, 136.21batch/s, acc=0.773, floss=0.331, hloss=0.224, loss=0.335]
Epoch 0@0| val: 100%|██████████| 11/11 [00:00<00:00, 138.84batch/s, acc=0.773, floss=0.331, hloss=0.224, loss=0.335]
Epoch 0@0| val: 100%|██████████| 11/11 [00:00<00:00, 145.74batch/s, acc=0.773, floss=0.331, hloss=0.224, loss=0.332]
Epoch 0@0| val: 100%|██████████| 11/11 [00:00<00:00, 53.26batch/s, acc=0.773, floss=0.33, hloss=0.224, loss=0.338]
Epoch 0@0| val: 100%|██████████| 11/11 [00:00<00:00, 145.19batch/s, acc=0.773, floss=0.33, hloss=0.224, loss=0.338]
Epoch 0@0| val: 100%|██████████| 11/11 [00:00<00:00, 144.57batch/s, acc=0.773, floss=0.33, hloss=0.224, loss=0.332]
Epoch 0@0| val: 100%|██████████| 11/11 [00:00<00:00, 146.95batch/s, acc=

In [6]:
print(eval_latent)
print(eval_logits)
print(eval_unique)
print(eval_conf)

{'0_foreign_loss': 0.3300317585468292, '1_home_loss': 0.22435645081780173, '2_no2home_loss': 0.335044648159634, '3_no2foreign_loss': 0.4728374559770931}
{'4_home_logits_loss': 0.37787170030853967, '5_foreign_logits_loss': 2.5348691257563503, '6_no2home_logits_loss': 1.0403578064658425, '7_home_acc': 0.8735795454545446, '8_foreign_acc': 0.44281249999999994, '9_no2home_acc': 0.6718750000000002}
{'fguide_hunique': 0.21871049891818653, 'fguide_funique': 1.3295947139913387, 'hguide_hunique': 1.811410093307495, 'hguide_funique': 0.014170067335097966}
{'confident_fguide_hunique': 2.734090909090909, 'confident_fguide_funique': 13.054545454545455, 'confident_hguide_hunique': 34.99545454545454, 'confident_hguide_funique': 0.0}
